[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [21]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = self.d_model // self.num_heads
        self.W_q = nn.Linear(self.d_model, self.d_model)
        self.W_k = nn.Linear(self.d_model, self.d_model)
        self.W_v = nn.Linear(self.d_model, self.d_model)
        self.W_o = nn.Linear(self.d_model, self.d_model)
        # pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
        q,k,v = self.W_q(x_q), self.W_k(x_kv), self.W_v(x_kv)
        bs,seq_len,_ = q.shape
        bs_k,seq_len_k,_ = k.shape
        q = q.reshape(bs, seq_len, self.num_heads, self.d_model // self.num_heads).transpose(1,2)
        k = k.reshape(bs_k, seq_len_k, self.num_heads, self.d_model // self.num_heads).transpose(1,2)
        v = v.reshape(bs_k, seq_len_k, self.num_heads, self.d_model // self.num_heads).transpose(1,2)
        attn_score = torch.softmax(q@k.transpose(2,3)/math.sqrt(self.d_k), dim=-1)
        attn_output = attn_score @ v
        return attn_output.reshape(bs, seq_len, self.d_model)
        # pass  # Q from x_q, K/V from x_kv, no causal mask

In [22]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [23]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.8ms)
  ✅ [2/4] Q and KV different lengths (1.3ms)
  ✅ [3/4] No causal mask — all KV affects all Q (38.9ms)
  ✅ [4/4] Gradient flow (20.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (62.2ms total)
  Progress saved. Run status() to see your dashboard.

